# Factorio Pyanodons Neutron Processing

Calculation for the *Neutron Capture* process for processing Plutonium by eliminating the waste products and processing them into wanted products.

This code is MIT-Licensed. See [`LICENSE.txt`](LICENSE.txt).

In [ ]:
import numpy as np
from scipy.optimize import linprog

## *Neutron Capture* Process Definition

How many times must each *Neutron Capture* process run to eliminate all the unwanted products?

x0
: Neutron Absorption PU-240 (239/241 -> 240/242)

x1
: Neutron Absorption PU-238 (239/240 -> 238/242)

x2
: Neutron Absorption PU-241 (2x242 -> 241/239)

x3
: Neutron Absorption PU-239 (241/242 -> 239/240)

(All values are in percent, i.e. multiplied by 100)


In [ ]:
# Process definitions.
# Index: 238/239/240/241/242.
# Positive: output, Negative: input
nc_240 = [0, -1, 1, 0, -1]
nc_238 = [1, -1, -1, 0, 1]
nc_241 = [0, 1, 0, 1, -2]
nc_239 = [0, 1, 1, -1, -1]
# Production matrix
nc = np.array([nc_240, nc_238, nc_241, nc_239])
# Production matrix transposed
nc_t = nc.transpose()
# Duration of each process in seconds
times = np.array([442, 202, 281, 552])


## *Neutron Capture* Process Output

PU-238 und PU-239 are useful.
PU-240, PU-241, PU-242 need to be removed.

## The Equation system

From the process definition we can derive the following system of equalities:

PU-238
:  2 +  0*x0 +  1*x1 + 0*x2 + 0*x3 = a, a >= 0

    0*x0 + -1*x1 + 0*x2 + 0*x3 <= 2

PU-239
: 53 + -1*x0 + -1*x1 + 1*x2 + 1*x3 = b, b >= 0

    1*x0 + 1*x1 + -1*x2 + -1*x3 <= 53

PU-240
: 25 + +1*x0 + -1*x1 + 0*x2 + 1*x3 = 0

    1*x0 + -1*x1 + 0*x2 + 1*x3 = -25

PU-241
: 15 + -1*x0 + 0*x1 + 1*x2 + -1*x3 = 0

    -1*x0 + 0*x1 + 1*x2 + -1*x3 = -15

PU-242
: 50 + 1*x0 + 1*x1 + -2*x2 + -1*x3 = 0 

    1*x0 + 1*x1 + -2*x2 + -1*x3 = -50


The equation system can be solved by using a linear programming minimization algorithm.

In [ ]:
# Output of the PU-Oxide procss. This is the input to the neutron capture process.
puox = np.array([2, 53, 25, 15, 50])
# Build the equation system from the process definition.
pu238 = nc_t[0]
pu239 = nc_t[1]
pu240 = nc_t[2]
pu241 = nc_t[3]
pu242 = nc_t[4]
# print("pu238:", pu238)
# print("pu239:", pu239)
# print("pu240:", pu240)
# print("pu241:", pu241)
# print("pu242:", pu242)
a0 = pu238 * -1
a1 = pu239 * -1
A_ub = np.array([pu238 * -1, pu239 * -1])
b_ub = np.array([puox[0], puox[1]])
A_eq = np.array([pu240 * -1, pu241 * -1, pu242 * -1])
b_eq = np.array([puox[2], puox[3], puox[4]])
# Objective function: minimize the number of cycles
c = np.array([1, 1, 1, 1])
# print("A_ub:", A_ub)
# print("b_ub:", b_ub)
# print("A_eq:", A_eq)
# print("b_eq:", b_eq)
res_cycles = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq)
print("res.x:", res_cycles.x)
print("res.message:", res_cycles.message)
if res_cycles.success:
    print(f"PU-240 neutron capture: {res_cycles.x[0] * 2:3.0f} times")
    print(f"PU-238 neutron capture: {res_cycles.x[1] * 2:3.0f} times")
    print(f"PU-241 neutron capture: {res_cycles.x[2] * 2:3.0f} times")
    print(f"PU-239 neutron capture: {res_cycles.x[3] * 2:3.0f} times")
    cycles = res_cycles.x * 2
    total_time = np.dot(cycles, times)
    print(f"Total time: {total_time:3.0f} seconds")

res.x: [ 0.  77.5 37.5 52.5]
res.message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
PU-240 neutron capture:   0 times
PU-238 neutron capture: 155 times
PU-241 neutron capture:  75 times
PU-239 neutron capture: 105 times
Total time: 110345 seconds


### Calculate the optimum number of machines

Again, a minimization problem:

Minimize the total time using the number of cycles calculated in the last step.

Constraint is, that each process has to be run the number of cycles from the last step.

To avoid the trivial solution (add a number of machines corresponding to the number of cycles), constrain the number of machines allowed for each process.

In [ ]:
# Objective function: minimize the total time
c = np.array([times[0], times[1], times[2], times[3]])
# Constraints: The number of cycles from the last step.
A_eq = np.array(
    [
        [times[0], 0, 0, 0],  # PU-240
        [0, times[1], 0, 0],  # PU-238
        [0, 0, times[2], 0],  # PU-241
        [0, 0, 0, times[3]],  # PU-239
    ]
)
b_eq = np.array([cycles[0], cycles[1], cycles[2], cycles[3]])
# Constraints: Limit the number of machines for each process.
A_ub = np.array(
    [
        [1, 0, 0, 0],  # PU-240
        [0, 1, 0, 0],  # PU-238
        [0, 0, 1, 0],  # PU-241
        [0, 0, 0, 1],  # PU-239
    ]
)
b_ub = np.array([16, 16, 16, 16])  # Limit the number of machines to 8 for each process.
res_machines = linprog(c, A_eq=A_eq, b_eq=b_eq, A_ub=A_ub, b_ub=b_ub)
print("res_machines.x:", res_machines.x)
print("res_machines.message:", res_machines.message)
if res_machines.success:
    # print(f"PU-240 machines: {res_machines.x[0]:f}")
    # print(f"PU-238 machines: {res_machines.x[1]:f}")
    # print(f"PU-241 machines: {res_machines.x[2]:f}")
    # print(f"PU-239 machines: {res_machines.x[3]:f}")
    # Round up the number of machines to the next integer.
    factor = 1.0 / np.min(np.extract(res_machines.x > 0, res_machines.x))
    # print(f"Factor to scale machines: {factor}")
    machines = np.ceil(res_machines.x * factor)
    # print(f"Number of machines: {machines}")
    print(f"PU-240 machines: {machines[0]:.0f}")
    print(f"PU-238 machines: {machines[1]:.0f}")
    print(f"PU-241 machines: {machines[2]:.0f}")
    print(f"PU-239 machines: {machines[3]:.0f}")
    total_time = np.dot(machines, times)
    print(f"Total time with machines: {total_time:3.0f} seconds")

res_machines.x: [0.         0.76732673 0.26690391 0.19021739]
res_machines.message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
PU-240 machines: 0
PU-238 machines: 5
PU-241 machines: 2
PU-239 machines: 1
Total time with machines: 2124 seconds
